# Week 2: Dataset Exploration & Logistic Regression Evaluation

This notebook covers **Task 1** (Exploratory Data Analysis) and **Task 4** (Logistic Regression Classification and Evaluation Metrics).

## Task 1 — Dataset Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# 1. Load the dataset
raw_data = load_breast_cancer()
df = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)
df['target'] = raw_data.target

# Inspect shape, columns, and data types
print(f"Dataset Shape: {df.shape}")
print("\nData Types and Info:")
print(df.info())

# 2. Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum().sum())

In [ ]:
# 3. Plot distributions of at least 3 features
features_to_plot = ['mean radius', 'mean texture', 'mean concavity']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(features_to_plot):
    sns.histplot(data=df, x=col, hue='target', kde=True, ax=axes[i], palette='coolwarm', bins=30)
    axes[i].set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

### Task 1 Strategy & Reasoning

```python
# Target Variable: 'target' (Binary classification: 0 for Malignant, 1 for Benign)
# Feature Variables: All 30 continuous structural metrics (radius, texture, smoothness, perimeter, etc.)
# 
# Missing Values Handling Strategy:
# This clean baseline dataset contains 0 missing entries. However, if missing items were encountered, 
# continuous structural variables would be imputed using their localized column median value to preserve structural 
# integrity against outliers. Rows lacking target variables would be immediately dropped.
# 
# Features Analysis and Usefulness:
# Features like 'mean radius', 'mean perimeter', and 'mean concavity' show distinct statistical separations 
# across target labels in histograms. They provide strong predictive utility because cellular mass expansion 
# and explicit shape distortions correlate highly with structural tissue variations.
```

## Task 4 — Logistic Regression + Evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

# Split features and target
X = df.drop(columns=['target'])
y = df['target']

# Split into 80% train and 20% test partitions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit Logistic Regression model
clf = LogisticRegression(max_iter=10000, random_state=42)
clf.fit(X_train, y_train)

# Predict
y_pred = clf.predict(X_test)

# Performance evaluation tracking reports
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy Score:  {acc:.4f}")
print(f"Precision Score: {prec:.4f}")
print(f"Recall Score:    {rec:.4f}")

In [ ]:
# Plot the confusion matrix as a heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Malignant (0)', 'Benign (1)'], 
            yticklabels=['Malignant (0)', 'Benign (1)'])
plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### Task 4 Reflection: Is accuracy enough to evaluate this model?

**Answer:** 
No, raw accuracy is insufficient for medical screening tasks. Accuracy treats all prediction errors equally, whereas a False Negative (misclassifying a malignant case as benign) is far more critical than a False Positive. Missing an active malignancy delays life-saving treatment. Therefore, monitoring **Recall** (Sensitivity) is vital to ensure that the model successfully captures as many true positive malignant cases as possible, even if accuracy looks high on paper.